# SageMaker: DistilBERT Text Classification (Online or Offline Weights)

**What this notebook does**
- Reads one or more CSVs from S3 (`text,label`)
- Creates stratified train/val/test splits
- Tokenizes with `AutoTokenizer`
- Trains DistilBERT using `Trainer`
- Saves model/tokenizer + label map locally and uploads back to S3

**Two modes for model weights**
1. **Online** (default): `MODEL_ID = 'distilbert-base-uncased'` (requires egress)
2. **Offline**: point `MODEL_S3_PREFIX` at a folder in S3 containing the model files
   (e.g., `config.json`, `pytorch_model.bin`, `tokenizer.json`, `vocab.txt`, etc.). The notebook will
   download them to a local dir and load from there.

**Assumptions**
- Running in SageMaker with an IAM role that can read/write to your S3 bucket.

In [ ]:
#!pip install -q boto3 s3fs transformers datasets evaluate pandas scikit-learn
import os, sys, json, time
from datetime import datetime
import boto3
import s3fs
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate


In [ ]:
# =====================
# CONFIG — EDIT ME
# =====================
CONFIG = {
    'AWS_REGION': os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'),
    'S3_BUCKET': 'your-bucket-name',                # <-- change
    'S3_INPUT_PREFIX': 'datasets/text/',            # CSVs live here
    'S3_OUTPUT_PREFIX': f'model-artifacts/distilbert/{datetime.utcnow().strftime("%Y%m%d-%H%M%S")}/',
    'S3_GLOB': '*.csv',
    'RANDOM_STATE': 42,
    'MAX_LENGTH': 256,
    'BATCH_SIZE': 16,
    'EPOCHS': 2,
    'LEARNING_RATE': 2e-5,
    # ---- Model source ----
    'MODEL_MODE': 'online',              # 'online' or 'offline'
    'MODEL_ID': 'distilbert-base-uncased',  # used if MODEL_MODE='online'
    'MODEL_S3_PREFIX': 'models/distilbert-base-uncased/',  # used if MODEL_MODE='offline'
}
CONFIG

In [ ]:
# -----------------
# S3 helpers
# -----------------
s3 = boto3.client('s3', region_name=CONFIG['AWS_REGION'])
fs = s3fs.S3FileSystem(anon=False)

def s3_uri(bucket, key):
    return f's3://{bucket}/{key}'

def list_s3_csvs(bucket, prefix, pattern='*.csv'):
    base = s3_uri(bucket, prefix)
    return fs.glob(f'{base}/**/{pattern}') if pattern else fs.glob(f'{base}/**/*.csv')

def upload_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)
    return s3_uri(bucket, key)

def download_s3_prefix(bucket, prefix, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith('/'):
                continue
            rel = key[len(prefix):].lstrip('/')
            dest = os.path.join(local_dir, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            s3.download_file(bucket, key, dest)
    return local_dir


In [ ]:
# -----------------
# Load dataset(s)
# -----------------
csv_paths = list_s3_csvs(CONFIG['S3_BUCKET'], CONFIG['S3_INPUT_PREFIX'], CONFIG['S3_GLOB'])
assert csv_paths, f'No CSVs found at s3://{CONFIG["S3_BUCKET"]}/{CONFIG["S3_INPUT_PREFIX"]} matching {CONFIG["S3_GLOB"]}'
dfs = [pd.read_csv(p, storage_options={'anon': False})[['text','label']] for p in csv_paths]
df = pd.concat(dfs, axis=0, ignore_index=True)
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 2].dropna(subset=['text','label'])
df = df.drop_duplicates(subset=['text','label'])
print('After hygiene:', df.shape)

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
print('Classes:', list(le.classes_))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=CONFIG['RANDOM_STATE'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=CONFIG['RANDOM_STATE'])

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text','label_id']], preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['text','label_id']], preserve_index=False),
    'test': Dataset.from_pandas(test_df[['text','label_id']], preserve_index=False),
})
ds

In [ ]:
# -----------------
# Model source (online vs offline)
# -----------------
if CONFIG['MODEL_MODE'] == 'online':
    model_id_or_path = CONFIG['MODEL_ID']
else:
    local_model_dir = 'offline_model'
    download_s3_prefix(CONFIG['S3_BUCKET'], CONFIG['MODEL_S3_PREFIX'], local_model_dir)
    model_id_or_path = local_model_dir

from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_id_or_path)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=CONFIG['MAX_LENGTH'])

ds_tok = ds.map(tok, batched=True, remove_columns=['text'])
ds_tok = ds_tok.rename_column('label_id','labels')
ds_tok.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_id_or_path, num_labels=num_labels)
num_labels

In [ ]:
acc = evaluate.load('accuracy')
f1m = evaluate.load('f1')

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        'accuracy': acc.compute(references=p.label_ids, predictions=preds)['accuracy'],
        'f1_macro': f1m.compute(references=p.label_ids, predictions=preds, average='macro')['f1']
    }

args = TrainingArguments(
    output_dir='distilbert_clf',
    per_device_train_batch_size=CONFIG['BATCH_SIZE'],
    per_device_eval_batch_size=CONFIG['BATCH_SIZE'],
    num_train_epochs=CONFIG['EPOCHS'],
    learning_rate=CONFIG['LEARNING_RATE'],
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to=[],
    seed=CONFIG['RANDOM_STATE'],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Evaluate on the test set
test_metrics = trainer.evaluate(ds_tok['test'])
test_metrics

In [ ]:
# Save locally and upload to S3
ART_DIR = f'distilbert_artifacts_{datetime.utcnow().strftime("%Y%m%d-%H%M%S")}'
os.makedirs(ART_DIR, exist_ok=True)
trainer.model.save_pretrained(f'{ART_DIR}/best')
tokenizer.save_pretrained(f'{ART_DIR}/best')
pd.Series(le.classes_).to_csv(f'{ART_DIR}/label_map.csv', index_label='id', header=['label'])
print('Saved locally to', ART_DIR)

uploaded = []
for fname in ['label_map.csv']:
    key = CONFIG['S3_OUTPUT_PREFIX'] + fname
    uploaded.append(upload_to_s3(f'{ART_DIR}/{fname}', CONFIG['S3_BUCKET'], key))

# Upload the model directory recursively
def upload_dir(local_dir, bucket, prefix):
    for root, dirs, files in os.walk(local_dir):
        for f in files:
            lp = os.path.join(root, f)
            rel = os.path.relpath(lp, local_dir).replace('\\','/')
            key = prefix.rstrip('/') + '/' + rel
            s3.upload_file(lp, bucket, key)
    return s3_uri(bucket, prefix)

uploaded_model_uri = upload_dir(f'{ART_DIR}/best', CONFIG['S3_BUCKET'], CONFIG['S3_OUTPUT_PREFIX']+'best')
uploaded, uploaded_model_uri